# Big Brother: Análisis Aumentado con IA

Demo de un sistema que compara análisis superficial vs análisis profundo (con datos ocultos) para detectar casos mal clasificados.

---

**Stack**: Python 3.12 · pandas · Faker · rich · openai (NVIDIA NIM)

**Datos**: 50 registros sintéticos (Faker), 18 casos donde `score_aparente != score_real`

---

In [ ]:
import pandas as pd
import json
from pathlib import Path

DATA_DIR = Path('..') / 'datos'

## Utilidades de análisis

In [ ]:
ALERTAS_PELIGRO = [
    "inmunosupresión", "inmunosuprimido", "quimioterapia", "trasplante",
    "anticoagulado", "warfarina", "aneurisma", "disección aórtica",
    "embarazo ectópico", "fractura patológica", "compromiso medular",
    "fuga anastomótica", "cardiopatía congénita", "Tetralogía de Fallot",
    "crisis hipoxémica", "taquicardia ventricular", "marcapasos",
    "síncope", "diabetes tipo 1", "cetoacidosis", "ERC estadio 4",
    "hiperpotasemia", "aneurisma cerebral", "QT largo", "muerte súbita",
    "sepsis", "neutropenia", "TVP", "TEP", "sangrado intracraneal",
    "torsión testicular", "brote lúpico", "nefritis", "VIH", "CD4",
    "infección oportunista", "paro cardíaco", "shock",
]


def analizar_paciente(row):
    notas = str(row.get("notas_relevantes", "")).lower()
    comorbilidades = str(row.get("comorbilidades", "")).lower()
    medicacion = str(row.get("medicacion_cronica", "")).lower()
    texto = f"{notas} {comorbilidades} {medicacion}"

    alertas = [a for a in ALERTAS_PELIGRO if a in texto]
    esi_aparente = int(row["esi_aparente"])
    esi_real = int(row["esi_real"])

    if alertas and esi_aparente > 2:
        esi_predicho = 2
    else:
        esi_predicho = esi_aparente

    es_trampa = esi_aparente != esi_predicho
    severidad = len(alertas)

    if severidad >= 5:
        impacto = "Crítico — Múltiples factores"
    elif severidad >= 2:
        impacto = "Significativo — Varios factores"
    elif severidad >= 1:
        impacto = "Moderado — Factor presente"
    else:
        impacto = "Ninguno"

    return {
        "id": int(row["id"]),
        "nombre": row["nombre"],
        "fuente": "Rule-based",
        "esi_aparente": esi_aparente,
        "esi_real": esi_real,
        "analisis_superficial_score": esi_aparente,
        "analisis_superficial_justificacion": f"Basado solo en síntoma: '{row['sintomas_actuales']}'",
        "analisis_superficial_info_usada": [f"Síntoma: {row['sintomas_actuales']}"],
        "analisis_superficial_info_ignorada": ["Historia completa"],
        "analisis_profundo_score": esi_predicho,
        "analisis_profundo_justificacion": f"Factores detectados: {', '.join(alertas)}" if alertas else "Sin factores",
        "analisis_profundo_factores": alertas[:5],
        "analisis_profundo_senhal": f"{len(alertas)} señales ocultas" if alertas else "Ninguna",
        "diferencia_cambio": f"Sube de {esi_aparente} a {esi_predicho}" if es_trampa else "Sin cambio",
        "diferencia_gravedad": "Crítico" if (es_trampa and esi_predicho <= 2) else "Significativo" if es_trampa else "Ninguno",
        "diferencia_impacto": impacto,
        "alerta_trampa": es_trampa,
        "esi_predicho": esi_predicho,
    }


def analizar_batch(df):
    return pd.DataFrame([analizar_paciente(row) for _, row in df.iterrows()])


def resumen(resultados):
    total = len(resultados)
    trampas = resultados["alerta_trampa"].sum()
    aciertos = (resultados["esi_predicho"] == resultados["esi_real"]).sum()
    print(f"Total: {total} · Discrepancias: {trampas} · Aciertos: {aciertos}/{total} ({100*aciertos/total:.0f}%)")
    for _, r in resultados[resultados["alerta_trampa"]].iterrows():
        print(f"  {r['nombre']:<25} Superficial: {int(r['analisis_superficial_score'])} → Profundo: {int(r['analisis_profundo_score'])} [{r['diferencia_gravedad']}]")

## 1. Cargar datos

In [ ]:
df_pacientes = pd.read_csv(DATA_DIR / 'pacientes.csv')
df_historias = pd.read_csv(DATA_DIR / 'historias_clinicas.csv')
df_guia = pd.read_csv(DATA_DIR / 'guia_triage_esi.csv')

df = df_pacientes.merge(df_historias, on='id', how='left')
casos = df[df['esi_aparente'] != df['esi_real']]

print(f'{len(df)} registros · {len(casos)} con discrepancia ({len(casos)/len(df)*100:.0f}%)')
df_guia

## 2. Discrepantes

In [ ]:
casos[['id','nombre','edad','sintomas_actuales','esi_aparente','esi_real','comorbilidades']]

## 3. Análisis batch

In [ ]:
resultados = analizar_batch(df)
resumen(resultados)

## 4. Caso detallado

In [ ]:
ejemplo = casos.iloc[0]
r = analizar_paciente(ejemplo)
print(json.dumps({
    'nombre': r['nombre'],
    'analisis_superficial': {
        'score': r['analisis_superficial_score'],
        'justificacion': r['analisis_superficial_justificacion'],
        'info_usada': r['analisis_superficial_info_usada'],
        'info_ignorada': r['analisis_superficial_info_ignorada'],
    },
    'analisis_profundo': {
        'score': r['analisis_profundo_score'],
        'justificacion': r['analisis_profundo_justificacion'],
        'factores': r['analisis_profundo_factores'],
        'senhal': r['analisis_profundo_senhal'],
    },
    'diferencia': {
        'cambio': r['diferencia_cambio'],
        'gravedad': r['diferencia_gravedad'],
        'impacto': r['diferencia_impacto'],
    }
}, indent=2, ensure_ascii=False))